# CD115-positive CDP differential expression and enrichment

Run cells from top to bottom. Paths and thread counts are centralized in `config/`; embedded plots are retained as visual references from the source analysis.


In [ ]:
# Centralized paths and scheduler-aware thread limits.
repo_root <- if (file.exists("config/paths.R")) "." else if (file.exists("../config/paths.R")) ".." else stop("Run Jupyter from the repository root or notebooks/ directory.")
source(file.path(repo_root, "config", "paths.R"))


In [1]:
# --- 1. 加载所有必需的 R 包 ---
#----------------------------------------------------
suppressPackageStartupMessages({
    library(dplyr)
    library(Seurat)
    library(clusterProfiler)
    library(org.Mm.eg.db) # 小鼠物种注释包
    library(enrichplot)
    library(msigdbr)
    library(writexl)
    library(KEGGREST)
    library(tidyr)
    library(tibble)
    # 自定义一个 "not in" 操作符，方便使用
    `%||%` <- function(a, b) if (is.null(a)) b else a
})

# --- 2. 设置路径和核心参数 ---
#----------------------------------------------------
# !! 您只需要修改下面这 3 行 !!
base_output_dir <- legacy_path("20250604p38-afterYZ/passway-gene") # 基础输出目录
seu_path        <- legacy_path("20250604p38-afterYZ/anno.rds")     # Seurat对象路径
target_celltype <- Sys.getenv("P38_TARGET_CELLTYPE", "4 CD115+ CDP")                                              # 核心参数：目标细胞亚群

# --- 自动准备 ---
# 从细胞类型名称创建一个对文件名安全的字符串 (移除空格、斜杠、加号)
filename_safe_celltype <- gsub("[ /+]+", "", target_celltype)

# 自动构建完整的输出目录
output_dir <- file.path(base_output_dir, filename_safe_celltype)
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

cat("目标细胞亚群:", target_celltype, "\n")
cat("完整输出目录已自动设置为:", output_dir, "\n\n")

# 读取 Seurat 对象
seu <- readRDS(seu_path)

# 提取目标细胞亚群 (使用参数)
PDC <- subset(seu, subset = celltype %in% target_celltype)
PDC$celltype <- droplevels(PDC$celltype)
cat("细胞亚群提取完成。细胞数:", ncol(PDC), "\n")

# 设置分组信息
Idents(PDC) <- "orig.ident"

# 运行 JoinLayers
tryCatch({
  if (length(Assays(PDC)) > 0) {
     DefaultAssay(PDC) <- Assays(PDC)[1]
     PDC <- JoinLayers(PDC)
     cat("JoinLayers 运行成功。\n")
  } else { warning("未找到任何 Assay，跳过 JoinLayers。") }
}, error = function(e) { cat("运行 JoinLayers 时出错:", conditionMessage(e), "\n") })


# --- 3. 差异基因表达分析 (DEG) ---
#----------------------------------------------------
cat("正在进行差异基因分析 (KO vs WT)...\n")
deg_result <- FindMarkers(
  object = PDC, ident.1 = "KO", ident.2 = "WT", min.pct = 0.25,
  logfc.threshold = 0.25, test.use = "wilcox", assay = DefaultAssay(PDC), verbose = FALSE
)

deg_sig_logfc <- 0.5
deg_sig_padj <- 0.05

deg_result <- deg_result %>%
  mutate(expression_level = case_when(
    avg_log2FC >= deg_sig_logfc & p_val_adj <= deg_sig_padj ~ "Upregulated",
    avg_log2FC <= -deg_sig_logfc & p_val_adj <= deg_sig_padj ~ "Downregulated",
    TRUE ~ "Not significant"
  )) %>%
  rownames_to_column(var = "gene") %>%
  dplyr::select(gene, avg_log2FC, p_val, p_val_adj, pct.1, pct.2, expression_level, everything())

cat("DEG 结果已生成。分类如下:\n")
print(table(deg_result$expression_level))

# 保存 DEG 结果到 Excel 文件 (使用动态文件名)
deg_output_file <- file.path(output_dir, paste0(filename_safe_celltype, "-DEG.xlsx"))
tryCatch({
    writexl::write_xlsx(list("DEG_Results" = deg_result), path = deg_output_file)
    cat("✅ 差异基因结果已保存至:", deg_output_file, "\n\n")
}, error=function(e){cat("保存 DEG.xlsx 失败:", conditionMessage(e),"\n")})


# --- 4. GSEA (基因集富集分析) ---
#----------------------------------------------------
run_gsea_analysis <- function(deg_result, pval_cutoff = 0.1) {
  cat("--- 开始运行 GSEA 分析 ---\n")
  gene_list <- deg_result$avg_log2FC; names(gene_list) <- deg_result$gene
  gene_list <- sort(gene_list[!is.na(gene_list)], decreasing = TRUE)
  if(length(gene_list) == 0) {cat("错误: GSEA 的基因列表为空。\n"); return(NULL)}
  gsea_results <- list(); gsea_min_size = 15; gsea_max_size = 500
  h_gene_set <- msigdbr(species = "Mus musculus", category = "H") %>% dplyr::select(gs_name, gene_symbol)
  kegg_gene_set <- msigdbr(species = "Mus musculus", category = "C2", subcategory = "CP:KEGG") %>% dplyr::select(gs_name, gene_symbol)
  gobp_gene_set <- msigdbr(species = "Mus musculus", category = "C5", subcategory = "GO:BP") %>% dplyr::select(gs_name, gene_symbol)
  gsea_results$hallmark <- tryCatch(GSEA(gene_list, TERM2GENE=h_gene_set, minGSSize=gsea_min_size, maxGSSize=gsea_max_size, pvalueCutoff=pval_cutoff, pAdjustMethod="BH", verbose=F), error=function(e) NULL)
  gsea_results$kegg <- tryCatch(GSEA(gene_list, TERM2GENE=kegg_gene_set, minGSSize=gsea_min_size, maxGSSize=gsea_max_size, pvalueCutoff=pval_cutoff, pAdjustMethod="BH", verbose=F), error=function(e) NULL)
  gsea_results$gobp <- tryCatch(GSEA(gene_list, TERM2GENE=gobp_gene_set, minGSSize=gsea_min_size, maxGSSize=gsea_max_size, pvalueCutoff=pval_cutoff, pAdjustMethod="BH", verbose=F), error=function(e) NULL)
  return(gsea_results)
}

save_gsea_results <- function(gsea_results, output_dir, filename_prefix) {
  if(is.null(gsea_results) || length(gsea_results) == 0) return(NULL)
  result_list <- list()
  if(!is.null(gsea_results$hallmark) && nrow(gsea_results$hallmark@result) > 0) result_list$hallmark <- as.data.frame(gsea_results$hallmark@result) %>% mutate(Category = "Hallmark")
  if(!is.null(gsea_results$kegg) && nrow(gsea_results$kegg@result) > 0) result_list$kegg <- as.data.frame(gsea_results$kegg@result) %>% mutate(Category = "KEGG")
  if(!is.null(gsea_results$gobp) && nrow(gsea_results$gobp@result) > 0) result_list$gobp <- as.data.frame(gsea_results$gobp@result) %>% mutate(Category = "GO_BP")
  if(length(result_list) == 0) return(NULL)
  all_gsea_results <- bind_rows(result_list) %>% arrange(desc(abs(NES)))
  output_file <- file.path(output_dir, paste0(filename_prefix, "-GSEA.xlsx"))
  tryCatch({
    writexl::write_xlsx(list("GSEA_All_Results" = all_gsea_results), path = output_file)
    cat("✅ 所有 GSEA 结果已合并并保存至:", output_file, "\n\n")
  }, error=function(e){cat("保存 GSEA.xlsx 失败:", conditionMessage(e),"\n")})
}

gsea_results <- run_gsea_analysis(deg_result, pval_cutoff = 0.2)
save_gsea_results(gsea_results, output_dir, filename_safe_celltype)


# --- 5. ORA (超几何富集分析) ---
#----------------------------------------------------
run_ora_analysis <- function(deg_result, direction, pval_cutoff = 0.05, qval_cutoff = 0.2) {
  cat(paste0("--- 运行 ORA (", direction, ") ---\n"))
  if (direction == "All") {diff_genes <- deg_result %>% filter(expression_level %in% c("Upregulated", "Downregulated")) %>% pull(gene)} else {diff_genes <- deg_result %>% filter(expression_level == direction) %>% pull(gene)}
  if(length(diff_genes) < 10) {cat("  基因过少，跳过分析。\n"); return(NULL)}
  gene_ids <- tryCatch(suppressMessages(bitr(diff_genes, "SYMBOL", "ENTREZID", org.Mm.eg.db)), error=function(e) NULL)
  if(is.null(gene_ids) || nrow(gene_ids) == 0) {cat("  ID转换失败，跳过分析。\n"); return(NULL)}
  
  enrichment_results <- list()
  enrichment_results$go_bp <- tryCatch(enrichGO(gene_ids$ENTREZID, org.Mm.eg.db, "ENTREZID", "BP", pAdjustMethod="BH", pvalueCutoff=pval_cutoff, qvalueCutoff=qval_cutoff), error=function(e) NULL)
  enrichment_results$go_cc <- tryCatch(enrichGO(gene_ids$ENTREZID, org.Mm.eg.db, "ENTREZID", "CC", pAdjustMethod="BH", pvalueCutoff=pval_cutoff, qvalueCutoff=qval_cutoff), error=function(e) NULL)
  enrichment_results$go_mf <- tryCatch(enrichGO(gene_ids$ENTREZID, org.Mm.eg.db, "ENTREZID", "MF", pAdjustMethod="BH", pvalueCutoff=pval_cutoff, qvalueCutoff=qval_cutoff), error=function(e) NULL)
  kegg_res <- tryCatch(enrichKEGG(gene_ids$ENTREZID, "mmu", "ncbi-geneid", pvalueCutoff=pval_cutoff, pAdjustMethod="BH", qvalueCutoff=qval_cutoff), error=function(e) NULL)
  if (!is.null(kegg_res) && nrow(kegg_res@result) > 0) {
      kegg_df <- as.data.frame(kegg_res@result)
      path_ids <- kegg_df$ID
      try({
          hierarchies <- sapply(path_ids, function(id) KEGGREST::keggGet(id)[[1]]$CLASS %||% "Unknown;Unknown")
          kegg_df <- kegg_df %>% mutate(Hierarchy = hierarchies) %>% separate(Hierarchy, into=c("CategoryA", "CategoryB"), sep=";\\s*", fill="right")
          kegg_res@result <- kegg_df
      })
  }
  enrichment_results$kegg <- kegg_res
  m_t2g <- msigdbr(species = "Mus musculus", category = "H") %>% dplyr::select(gs_name, gene_symbol)
  enrichment_results$hallmark <- tryCatch(enricher(diff_genes, TERM2GENE=m_t2g, pvalueCutoff=pval_cutoff, pAdjustMethod="BH", qvalueCutoff=qval_cutoff), error=function(e) NULL)
  return(enrichment_results)
}

standardize_and_save_ora <- function(ora_results, output_path, sheet_name = "ORA_Results") {
  if (is.null(ora_results) || length(ora_results) == 0) {cat(" ", sheet_name, "没有富集结果可保存。\n"); return(invisible(NULL))}
  result_list <- list()
  if(!is.null(ora_results$go_bp) && nrow(ora_results$go_bp) > 0) result_list$go_bp <- as.data.frame(ora_results$go_bp) %>% mutate(Category = "GO_BP")
  if(!is.null(ora_results$go_cc) && nrow(ora_results$go_cc) > 0) result_list$go_cc <- as.data.frame(ora_results$go_cc) %>% mutate(Category = "GO_CC")
  if(!is.null(ora_results$go_mf) && nrow(ora_results$go_mf) > 0) result_list$go_mf <- as.data.frame(ora_results$go_mf) %>% mutate(Category = "GO_MF")
  if(!is.null(ora_results$kegg) && nrow(ora_results$kegg) > 0) result_list$kegg <- as.data.frame(ora_results$kegg) %>% mutate(Category = "KEGG")
  if(!is.null(ora_results$hallmark) && nrow(ora_results$hallmark) > 0) result_list$hallmark <- as.data.frame(ora_results$hallmark) %>% mutate(Category = "Hallmark")
  if(length(result_list) == 0) {cat(" ", sheet_name, "所有 ORA 分析均未产生有效结果。\n"); return(invisible(NULL))}
  
  all_ora_results <- bind_rows(result_list)
  cat("  正在将ENTREZID转换为Gene Symbol...\n")
  rows_to_convert <- which(all_ora_results$Category %in% c("GO_BP", "GO_CC", "GO_MF", "KEGG"))
  if (length(rows_to_convert) > 0) {
    entrez_ids_to_convert <- unique(unlist(strsplit(all_ora_results$geneID[rows_to_convert], "/")))
    id_map <- suppressMessages(mapIds(org.Mm.eg.db, keys=entrez_ids_to_convert, column="SYMBOL", keytype="ENTREZID", multiVals="first"))
    all_ora_results$geneID[rows_to_convert] <- sapply(all_ora_results$geneID[rows_to_convert], function(id_string) {
      entrez_ids <- unlist(strsplit(id_string, "/")); symbols <- id_map[entrez_ids]
      paste(symbols[!is.na(symbols)], collapse = "/")
    })
  }
  all_ora_results <- all_ora_results %>% arrange(p.adjust)
  tryCatch({
    writexl::write_xlsx(setNames(list(all_ora_results), sheet_name), path = output_path)
    cat("✅ ORA 结果已保存至:", output_path, "\n\n")
  }, error = function(e){cat("保存", basename(output_path), "失败:", conditionMessage(e), "\n")})
}

# --- 6. 执行所有 ORA 分析并保存结果 ---
#----------------------------------------------------
# (1) 对所有差异基因进行 ORA
ora_all <- run_ora_analysis(deg_result, direction = "All")
standardize_and_save_ora(ora_all, file.path(output_dir, paste0(filename_safe_celltype, "-ORA.xlsx")), "ORA_All_DEGs")

# (2) 仅对上调基因进行 ORA
ora_up <- run_ora_analysis(deg_result, direction = "Upregulated")
standardize_and_save_ora(ora_up, file.path(output_dir, paste0(filename_safe_celltype, "-UpReg.xlsx")), "ORA_Upregulated")

# (3) 仅对下调基因进行 ORA
ora_down <- run_ora_analysis(deg_result, direction = "Downregulated")
standardize_and_save_ora(ora_down, file.path(output_dir, paste0(filename_safe_celltype, "-DownReg.xlsx")), "ORA_Downregulated")

cat("--- 所有分析流程已完成！---\n")

目标细胞亚群: 4 CD115+ CDP 
完整输出目录已自动设置为: /data3/Group8/gonglihao/20250604p38-afterYZ/passway-gene/4CD115CDP 

细胞亚群提取完成。细胞数: 1523 
JoinLayers 运行成功。
正在进行差异基因分析 (KO vs WT)...
DEG 结果已生成。分类如下:

  Downregulated Not significant     Upregulated 
             91            1425             180 
✅ 差异基因结果已保存至: /data3/Group8/gonglihao/20250604p38-afterYZ/passway-gene/4CD115CDP/4CD115CDP-DEG.xlsx 

--- 开始运行 GSEA 分析 ---


Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“For some of the pathways the P-values were likely overestimated. For such pathways log2err is set to NA.”
Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“For some pathways, in reality P-values are less than 1e-10. You can set the `eps` argument to zero for better estimation.”


✅ 所有 GSEA 结果已合并并保存至: /data3/Group8/gonglihao/20250604p38-afterYZ/passway-gene/4CD115CDP/4CD115CDP-GSEA.xlsx 

--- 运行 ORA (All) ---


Warning message in bitr(diff_genes, "SYMBOL", "ENTREZID", org.Mm.eg.db):
“4.06% of input gene IDs are fail to map...”
Reading KEGG annotation online: "https://rest.kegg.jp/link/mmu/pathway"...

Warning message in file(con, "r"):
“cannot open URL 'https://rest.kegg.jp/link/mmu/pathway': HTTP status was '403 Forbidden'”


  正在将ENTREZID转换为Gene Symbol...
✅ ORA 结果已保存至: /data3/Group8/gonglihao/20250604p38-afterYZ/passway-gene/4CD115CDP/4CD115CDP-ORA.xlsx 

--- 运行 ORA (Upregulated) ---


Warning message in bitr(diff_genes, "SYMBOL", "ENTREZID", org.Mm.eg.db):
“5% of input gene IDs are fail to map...”
Reading KEGG annotation online: "https://rest.kegg.jp/link/mmu/pathway"...

Reading KEGG annotation online: "https://rest.kegg.jp/list/pathway/mmu"...

Reading KEGG annotation online: "https://rest.kegg.jp/conv/ncbi-geneid/mmu"...



Error in .getUrl(url, .flatFileParser) : Forbidden (HTTP 403).
  正在将ENTREZID转换为Gene Symbol...
✅ ORA 结果已保存至: /data3/Group8/gonglihao/20250604p38-afterYZ/passway-gene/4CD115CDP/4CD115CDP-UpReg.xlsx 

--- 运行 ORA (Downregulated) ---


Warning message in bitr(diff_genes, "SYMBOL", "ENTREZID", org.Mm.eg.db):
“2.2% of input gene IDs are fail to map...”


Error in .getUrl(url, .flatFileParser) : Forbidden (HTTP 403).
  正在将ENTREZID转换为Gene Symbol...
✅ ORA 结果已保存至: /data3/Group8/gonglihao/20250604p38-afterYZ/passway-gene/4CD115CDP/4CD115CDP-DownReg.xlsx 

--- 所有分析流程已完成！---
